# Система компьютерного зрения: распознавание геометрических фигур
## Перебор гиперпараметров нейронной сети (треугольник, круг, квадрат)

In [1]:
# Подключение класса для создания нейронной сети прямого распространения
from tensorflow.keras.models import Sequential
# Подключение класса для создания полносвязного слоя
from tensorflow.keras.layers import Dense
# Подключение оптимизатора
from tensorflow.keras.optimizers import Adam
# Подключение утилит для to_categorical
from tensorflow.keras import utils
# Подключение библиотеки для загрузки изображений
from tensorflow.keras.preprocessing import image
# Подключение библиотеки для работы с массивами
import numpy as np
# Подключение модуля для работы с файлами
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
!pip install gdown -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Загрузка датасета из облака
import gdown
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l3/hw_light.zip', None, quiet=True)

# Распаковываем архив hw_light.zip
!unzip -q hw_light.zip

In [4]:
# Путь к директории с базой
base_dir = 'hw_light'
# Создание пустого списка для загрузки изображений обучающей выборки
x_train = []
# Создание списка для меток классов
y_train = []
# Задание высоты и ширины загружаемых изображений
img_height = 20
img_width = 20
# Перебор папок в директории базы
for patch in os.listdir(base_dir):
    # Перебор файлов в папках
    for img in os.listdir(base_dir + '/' + patch):
        # Добавление в список изображений текущей картинки
        x_train.append(image.img_to_array(image.load_img(base_dir + '/' + patch + '/' + img,
                                                          target_size=(img_height, img_width),
                                                          color_mode='grayscale')))
        # Добавление в массив меток, соответствующих классам
        if patch == '0':
            y_train.append(0)
        elif patch == '3':
            y_train.append(1)
        else:
            y_train.append(2)

# Преобразование в numpy-массив загруженных изображений и меток классов
x_train = np.array(x_train)
y_train = np.array(y_train)
# Вывод размерностей
print('Размер массива x_train', x_train.shape)
print('Размер массива y_train', y_train.shape)

Размер массива x_train (302, 20, 20, 1)
Размер массива y_train (302,)


In [5]:
# Нормализация пикселей (приводим к float32 и делим на 255)
x_train = x_train.astype('float32') / 255.0

# Разворачиваем изображения в вектор: из (302, 20, 20, 1) в (302, 400)
x_train_flat = x_train.reshape(x_train.shape[0], -1)

# Преобразование меток в one-hot encoding (3 класса)
num_classes = 3
y_train_cat = utils.to_categorical(y_train, num_classes)

print('x_train_flat shape:', x_train_flat.shape)
print('y_train_cat shape:', y_train_cat.shape)

x_train_flat shape: (302, 400)
y_train_cat shape: (302, 3)


In [6]:
# Параметры для перебора
neurons_list = [10, 100, 5000]
activations  = ['relu', 'linear']
batch_sizes  = [10, 100, 1000]

results   = []
input_dim = x_train_flat.shape[1]  # 400
epochs    = 10

combo_num = 0
for neurons in neurons_list:
    for activation in activations:
        for batch_size in batch_sizes:
            combo_num += 1

            # Создание модели
            model = Sequential([
                Dense(neurons, activation=activation, input_shape=(input_dim,)),
                Dense(num_classes, activation='softmax')
            ])

            model.compile(
                optimizer=Adam(),
                loss='categorical_crossentropy',
                metrics=['accuracy']
            )

            history = model.fit(
                x_train_flat, y_train_cat,
                epochs=epochs,
                batch_size=batch_size,
                verbose=0
            )

            final_acc = history.history['accuracy'][-1]
            print(f'[{combo_num}/18] Нейронов={neurons}, Активация={activation}, Batch={batch_size}  →  Точность: {final_acc:.4f}')

            results.append({
                'Нейронов': neurons,
                'Активация': activation,
                'Batch size': batch_size,
                'Точность': round(final_acc, 4)
            })

[1/18] Нейронов=10, Активация=relu, Batch=10  →  Точность: 0.8344
[2/18] Нейронов=10, Активация=relu, Batch=100  →  Точность: 0.6424
[3/18] Нейронов=10, Активация=relu, Batch=1000  →  Точность: 0.4371
[4/18] Нейронов=10, Активация=linear, Batch=10  →  Точность: 0.7152
[5/18] Нейронов=10, Активация=linear, Batch=100  →  Точность: 0.5695
[6/18] Нейронов=10, Активация=linear, Batch=1000  →  Точность: 0.4570
[7/18] Нейронов=100, Активация=relu, Batch=10  →  Точность: 0.8874
[8/18] Нейронов=100, Активация=relu, Batch=100  →  Точность: 0.7914
[9/18] Нейронов=100, Активация=relu, Batch=1000  →  Точность: 0.7483
[10/18] Нейронов=100, Активация=linear, Batch=10  →  Точность: 0.7980
[11/18] Нейронов=100, Активация=linear, Batch=100  →  Точность: 0.5993
[12/18] Нейронов=100, Активация=linear, Batch=1000  →  Точность: 0.5397
[13/18] Нейронов=5000, Активация=relu, Batch=10  →  Точность: 0.9702
[14/18] Нейронов=5000, Активация=relu, Batch=100  →  Точность: 0.8046
[15/18] Нейронов=5000, Активация=rel

In [6]:
# Сравнительная таблица результатов
df = pd.DataFrame(results)
df.index = range(1, len(df) + 1)
df.index.name = ''

print('Сравнительная таблица результатов:')
print(df.to_string())

Сравнительная таблица результатов:
    Нейронов Активация  Batch size  Точность
1         10      relu          10    0.8245
2         10      relu         100    0.5464
3         10      relu        1000    0.5464
4         10    linear          10    0.8278
5         10    linear         100    0.5762
6         10    linear        1000    0.5497
7        100      relu          10    0.9338
8        100      relu         100    0.7450
9        100      relu        1000    0.7384
10       100    linear          10    0.8808
11       100    linear         100    0.7450
12       100    linear        1000    0.5993
13      5000      relu          10    0.9007
14      5000      relu         100    0.8079
15      5000      relu        1000    0.4570
16      5000    linear          10    0.6623
17      5000    linear         100    0.5762
18      5000    linear        1000    0.4040
